# E2 — Cross-device transfer

Estimate performance when a classifier is deployed on a paired device, with and without a small labeled target-device calibration set. The devices also differ in stand, so the result is not an isolated chip effect.

## 1. Settings

Both transfer directions use identical target-label budgets (1, 5, or 10 examples per class). A separate three-gas multi-label task evaluates the second device pair.

In [ ]:
from pathlib import Path
import os
import copy
import random
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from esdata import EsmellDataset, Chain
from esdata.transforms import (
    DefaultCSVPreprocessing, SimplifyLabels, SelectRChannels,
    UserLabelCycleSegments, CoolingHeatingSegments,
    CanonicalTempAlign, CanonicalSubtractionFeatures,
)

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "src" / "esdata").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Benchmark repository root not found")
DATA_DIR = Path(os.environ.get("ESMELL_DATA_DIR", str(REPO_ROOT / "data" / "raw")))
OUT = REPO_ROOT / "outputs" / "E2_fewshot"
OUT.mkdir(parents=True, exist_ok=True)

ARCHIVES = {171: "curiosity-5v16.171.0.tar.gz", 172: "curiosity-1v17.172.0.tar.gz"}
MIX_ARCHIVES = {129: "curiosity-1v4.129.0.tar.gz", 130: "curiosity-2v15.130.0.tar.gz"}
CHANNELS = ("R1", "R2")                  # channels shared by both devices
BUDGETS = (1, 5, 10)                     # target labels per class
CALIB_SEEDS = (42, 43, 44)               # repeated calibration draws
RUN_MIXTURES = True                    # skipped with warning when data are missing
SPLIT_SEED = 42                          # one fixed matched-pair test
EPOCHS, PATIENCE = 100, 20              # source pretraining
ADAPT_EPOCHS = 25                        # fixed; no target validation
TARGET_ONLY_EPOCHS = 60                  # train CNN from scratch; no target validation
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Paired records

Use the original `esdata` pipeline on R1/R2. Match exposures by nominal label and time, and assign each pair to a single partition.

In [ ]:
HEAT, COOL = (1, -3, 5), (0, -2, 4)

def label_name(text):
    return str(text).split("[")[0].strip().lower().replace("amonia", "ammonia")


def load_card(card):
    source = EsmellDataset(root_path=str(DATA_DIR), subfolders=ARCHIVES[card],
                           csv_repair="strict", archive_cache="disk", preload=True)
    pipeline = source >> Chain(
        DefaultCSVPreprocessing(), SimplifyLabels(),
        SelectRChannels(include=CHANNELS),
        UserLabelCycleSegments(),
        CoolingHeatingSegments(temp_col="T_mean", smooth_window=3, eps=0.2),
        CanonicalTempAlign(n_grid=200, temp_range=(175., 390.),
                           normalize=False, conductance=True),
        CanonicalSubtractionFeatures(features=(HEAT, (HEAT, COOL))),
    )
    assert tuple(pipeline.resistance_channels) == CHANNELS
    rows, xs, errors = [], [], []
    for sid in tqdm(source.ids, desc=f"Card {card}"):
        info = source.metadata(sid)
        try:
            x = np.asarray(pipeline.features(sid), dtype=np.float32)
            if x.shape != (4, 200) or not np.isfinite(x).all() or not x.any():
                raise ValueError(f"Invalid tensor {x.shape}")
            rows.append({"id": str(info["filename"]),
                         "time": pd.Timestamp(info["datetime"]),
                         "gas": label_name(info["classname"]), "x_idx": len(xs)})
            xs.append(x)
        except Exception as exc:
            errors.append({"card": card, "id": str(info["filename"]),
                           "reason": str(exc)[:200]})
    print(f"{card}: {len(xs)} processed / {len(source.ids)} raw")
    return pd.DataFrame(rows), np.stack(xs), errors

cards = {}
qc = []
for card in ARCHIVES:
    info, array, failures = load_card(card)
    cards[card] = (info, array)
    qc.extend(failures)
pd.DataFrame(qc, columns=["card", "id", "reason"]).to_csv(OUT / "qc_errors.csv", index=False)

left = cards[171][0].rename(columns={"id": "id_171", "x_idx": "ix_171"}).sort_values("time")
right = cards[172][0].rename(columns={"id": "id_172", "x_idx": "ix_172"}).sort_values("time")
paired = pd.merge_asof(left, right, on="time", by="gas", direction="nearest",
                       tolerance=pd.Timedelta(seconds=2)).dropna(subset=["ix_172"]).copy()
paired["ix_171"] = paired.ix_171.astype(int)
paired["ix_172"] = paired.ix_172.astype(int)
paired = paired.sort_values(["time", "id_171"]).reset_index(drop=True)
paired.insert(0, "pair_id", [f"pair_{i:04d}" for i in range(len(paired))])
assert paired.id_171.is_unique and paired.id_172.is_unique, "Ambiguous pair matching"
assert len(paired) >= 600, "Too few matched pairs: inspect preprocessing and timestamps"

CLASSES = tuple(sorted(paired.gas.unique()))
assert len(CLASSES) == 7, f"Expected 7 classes, found {CLASSES}"
class_id = {name: i for i, name in enumerate(CLASSES)}
y = paired.gas.map(class_id).to_numpy(dtype=np.int64)
X = {card: cards[card][1][paired[f"ix_{card}"].to_numpy()] for card in ARCHIVES}
assert all(a.shape == (len(paired), 4, 200) for a in X.values())
paired.to_csv(OUT / "pairs.csv", index=False)
print("Matched pairs:", len(paired), "\n", paired.gas.value_counts().to_string())

## 3. Evaluation cohorts

Hold out 20% of matched pairs. Select source checkpoints from source-development records; select target calibration examples only from target development.

In [ ]:
all_pairs = np.arange(len(paired))
outer = StratifiedShuffleSplit(n_splits=1, test_size=.20, random_state=SPLIT_SEED)
dev_loc, test_loc = next(outer.split(all_pairs, y))
dev, test = all_pairs[dev_loc], all_pairs[test_loc]
inner = StratifiedShuffleSplit(n_splits=1, test_size=.25, random_state=SPLIT_SEED)
tr_loc, val_loc = next(inner.split(dev, y[dev]))
source_train, source_val = dev[tr_loc], dev[val_loc]
assert not (set(dev) & set(test)) and not (set(source_train) & set(source_val))
assert set(source_train) | set(source_val) == set(dev)
assert all(set(y[ix]) == set(range(7)) for ix in (source_train, source_val, test))

split_rows = []
for role, indices in (("source_train", source_train), ("source_val", source_val),
                      ("paired_test", test)):
    split_rows += [{"role": role, "seed": SPLIT_SEED, "budget_per_class": 0,
                    "pair_id": paired.pair_id.iloc[i]} for i in indices]

calibration = {}
for seed in CALIB_SEEDS:
    rng = np.random.default_rng(seed)
    order = {cls: rng.permutation(dev[y[dev] == cls]) for cls in range(7)}
    for budget in BUDGETS:
        chosen = np.sort(np.concatenate([order[cls][:budget] for cls in range(7)]))
        assert len(chosen) == budget * 7 and not (set(chosen) & set(test))
        calibration[seed, budget] = chosen
        split_rows += [{"role": "target_calibration", "seed": seed,
                        "budget_per_class": budget,
                        "pair_id": paired.pair_id.iloc[i]} for i in chosen]

pd.DataFrame(split_rows).to_csv(OUT / "splits.csv", index=False)
print(f"Train={len(source_train)} · source val={len(source_val)} · "
      f"target dev={len(dev)} · paired test={len(test)}")
print("Test classes:", dict(zip(CLASSES, np.bincount(y[test], minlength=7))))

## 4. Classifiers and calibration

Compare source-only, target-only, pooled-tree, and CNN fine-tuning approaches at matched target-label budgets. The full-target reference is not a few-shot method.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class EsmellEncoder(nn.Module):
    def __init__(self, channels=4):
        super().__init__()
        def block(a, b, kernel):
            return nn.Sequential(
                nn.Conv1d(a, b, kernel, padding=kernel // 2), nn.BatchNorm1d(b), nn.ReLU(),
                nn.Conv1d(b, b, kernel, padding=kernel // 2), nn.BatchNorm1d(b), nn.ReLU(),
                nn.MaxPool1d(2),
            )
        self.features = nn.Sequential(block(channels, 64, 7), block(64, 128, 5))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.projection = nn.Linear(128, 128)

    def forward(self, x):
        return self.projection(self.pool(self.features(x)).squeeze(-1))


class TransferCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EsmellEncoder(len(CHANNELS) * 2)
        self.classifier = nn.Sequential(nn.Linear(128, 128), nn.ReLU(),
                                        nn.Dropout(.2), nn.Linear(128, len(CLASSES)))

    def forward(self, x):
        return self.classifier(self.encoder(x))


def batches(x, indices, shuffle=False, seed=42, batch_size=64):
    return DataLoader(TensorDataset(torch.from_numpy(x[indices]),
                                    torch.from_numpy(y[indices])),
                      batch_size=batch_size, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed))


def predict(model, x, indices):
    model.eval()
    result = []
    with torch.no_grad():
        for xb, _ in batches(x, indices):
            result.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(result)


def source_pretrain(x, seed=42):
    set_seed(seed)
    model = TransferCNN().to(DEVICE)
    count = np.bincount(y[source_train], minlength=len(CLASSES))
    weights = torch.tensor(count.sum() / (len(CLASSES) * count),
                           dtype=torch.float32, device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = batches(x, source_train, shuffle=True, seed=seed)
    best_f1, stale, best_epoch, state = -1., 0, 0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(xb.to(DEVICE)), yb.to(DEVICE), weight=weights)
            loss.backward()
            opt.step()
        val_f1 = f1_score(y[source_val], predict(model, x, source_val), average="macro")
        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            state, stale = copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(state)
    return model, best_epoch, best_f1


def finetune(pretrained, x, fit, mode, seed):
    set_seed(seed)
    model = copy.deepcopy(pretrained).to(DEVICE)
    if mode == "head":
        model.encoder.requires_grad_(False)
    lr = 3e-4 if mode == "head" else 1e-4
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr,
                            weight_decay=1e-4)
    loader = batches(x, fit, shuffle=True, seed=seed, batch_size=32)
    for _ in range(ADAPT_EPOCHS):
        model.train()
        for layer in model.modules():
            if isinstance(layer, nn.BatchNorm1d):
                layer.eval()  # freeze running statistics for few-shot fine-tuning
        for xb, yb in loader:
            opt.zero_grad()
            nn.functional.cross_entropy(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
    return model


def target_from_scratch(x, fit, seed):
    set_seed(seed)
    model = TransferCNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = batches(x, fit, shuffle=True, seed=seed, batch_size=32)
    for _ in range(TARGET_ONLY_EPOCHS):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            nn.functional.cross_entropy(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
    return model


def summary_metrics(true, pred):
    return {"bacc": balanced_accuracy_score(true, pred),
            "macro_f1": f1_score(true, pred, average="macro", labels=range(7), zero_division=0)}


def tree_stats(x):
    blocks = [x.mean(2), x.std(2), x.min(2), x.max(2),
              np.median(x, axis=2), np.quantile(np.abs(x), .9, axis=2)]
    blocks.extend(b.mean(2) for b in np.array_split(x, 10, axis=2))
    return np.concatenate(blocks, axis=1)

## 5. Bidirectional evaluation

Run both source–target directions and retain every per-record prediction.

In [ ]:
results, predictions, pretrain_rows = [], [], []


def record(direction, method, budget, seed, fit_n, pred):
    truth = y[test]
    stats = summary_metrics(truth, pred)
    results.append({"direction": direction, "method": method,
                    "budget_per_class": budget, "seed": seed,
                    "n_target_labels": fit_n, "n_test": len(test), **stats})
    predictions.extend({"direction": direction, "method": method,
                        "budget_per_class": budget, "seed": seed,
                        "pair_id": paired.pair_id.iloc[i], "true": int(y[i]),
                        "pred": int(p)} for i, p in zip(test, pred))


for src, tgt in ((171, 172), (172, 171)):
    direction = f"{src}->{tgt}"
    print("\n", direction)
    base, epoch, val_f1 = source_pretrain(X[src], seed=SPLIT_SEED)
    source_bacc = balanced_accuracy_score(y[test], predict(base, X[src], test))
    record(direction, "source_only", 0, SPLIT_SEED, 0, predict(base, X[tgt], test))
    pretrain_rows.append({"direction": direction, "best_epoch": epoch,
                          "source_val_f1": val_f1, "source_test_bacc": source_bacc,
                          "target_test_bacc": results[-1]["bacc"]})
    print(f"Source val macro-F1={val_f1:.3f} (epoch {epoch}); "
          f"source-test BAcc={source_bacc:.3f}; target-test BAcc={results[-1]['bacc']:.3f}")
    torch.save({"state_dict": base.cpu().state_dict(), "classes": CLASSES,
                "source_chip": src, "best_epoch": epoch}, OUT / f"source_{src}.pt")
    base.to(DEVICE)

    z = tree_stats(X[tgt])
    z_src = tree_stats(X[src])
    source_trees = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=1,
                                       class_weight="balanced", random_state=SPLIT_SEED, n_jobs=-1)
    source_trees.fit(z_src[dev], y[dev])
    record(direction, "source_only_trees", 0, SPLIT_SEED, 0, source_trees.predict(z[test]))
    reference = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=1,
                                     class_weight="balanced", random_state=SPLIT_SEED, n_jobs=-1)
    reference.fit(z[dev], y[dev])
    record(direction, "target_reference", -1, SPLIT_SEED, len(dev), reference.predict(z[test]))

    for seed in CALIB_SEEDS:
        for budget in BUDGETS:
            fit = calibration[seed, budget]
            assert not np.intersect1d(fit, test).size
            for mode in ("head", "full"):
                model = finetune(base, X[tgt], fit, mode, seed)
                record(direction, f"finetune_{mode}", budget, seed, len(fit),
                       predict(model, X[tgt], test))
                del model
            fresh = target_from_scratch(X[tgt], fit, seed)
            record(direction, "target_only_cnn", budget, seed, len(fit),
                   predict(fresh, X[tgt], test))
            del fresh
            trees = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=1,
                                         class_weight="balanced", random_state=seed, n_jobs=-1)
            trees.fit(z[fit], y[fit])
            record(direction, "target_only_trees", budget, seed, len(fit), trees.predict(z[test]))
            pooled = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=1,
                                          class_weight="balanced", random_state=seed, n_jobs=-1)
            joined_x = np.concatenate([z_src[dev], z[fit]])
            joined_y = np.concatenate([y[dev], y[fit]])
            # Source and target each contribute half the total training weight.
            sample_w = np.concatenate([np.full(len(dev), .5 / len(dev)),
                                       np.full(len(fit), .5 / len(fit))])
            pooled.fit(joined_x, joined_y, sample_weight=sample_w)
            record(direction, "pooled_trees", budget, seed, len(fit), pooled.predict(z[test]))
            print(f"  seed {seed}, {budget}/class: done")

scores = pd.DataFrame(results)
preds = pd.DataFrame(predictions)
scores.to_csv(OUT / "scores.csv", index=False)
preds.to_csv(OUT / "predictions.csv", index=False)
pd.DataFrame(pretrain_rows).to_csv(OUT / "source_baselines.csv", index=False)
print("\n", scores.groupby(["direction", "method", "budget_per_class"])
      .bacc.agg(["mean", "std"]).round(3).to_string())

## 6. Primary transfer results

Export performance versus target-label budget and per-class results. Repeated calibration subsets reflect sampling variation, not independent devices.

In [ ]:
agg = (scores.groupby(["direction", "method", "budget_per_class"], as_index=False)
       .agg(bacc_mean=("bacc", "mean"), bacc_sd=("bacc", "std"),
            macro_f1_mean=("macro_f1", "mean"), n_target_labels=("n_target_labels", "first")))
agg.to_csv(OUT / "summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.1), sharey=True)
for ax, direction in zip(axes, ("171->172", "172->171")):
    sub = agg[agg.direction.eq(direction)]
    for method in ("finetune_head", "finetune_full", "target_only_cnn", "target_only_trees", "pooled_trees"):
        p = sub[sub.method.eq(method)].sort_values("budget_per_class")
        ax.errorbar(p.budget_per_class, p.bacc_mean, yerr=p.bacc_sd.fillna(0),
                    marker="o", capsize=3, label=method.replace("_", " "))
    for method, linestyle in (("source_only", "--"), ("source_only_trees", "-."), ("target_reference", ":")):
        value = sub[sub.method.eq(method)].bacc_mean.iloc[0]
        ax.axhline(value, ls=linestyle, label=method.replace("_", " "))
    ax.set(title=direction, xlabel="Target labels per class", xticks=BUDGETS, ylim=(0, 1.02))
    ax.grid(alpha=.2)
axes[0].set_ylabel("Balanced accuracy")
axes[1].legend(fontsize=7, loc="lower right", ncol=2)
fig.tight_layout()
fig.savefig(OUT / "fig_E2_fewshot.pdf", bbox_inches="tight")
fig.savefig(OUT / "fig_E2_fewshot.png", dpi=180, bbox_inches="tight")
plt.show()

class_recall = (preds.assign(correct=preds.true.eq(preds.pred))
                .groupby(["direction", "method", "budget_per_class", "seed", "true"], as_index=False)
                .agg(n=("correct", "size"), recall=("correct", "mean")))
class_recall["class"] = class_recall.true.map(dict(enumerate(CLASSES)))
class_recall.to_csv(OUT / "class_recalls.csv", index=False)

print("\nSaved to", OUT.resolve())
print("Overleaf §6 / E2: fig_E2_fewshot.pdf; summary.csv; class_recalls.csv.")

### Interpretation

Source-only performance measures zero-shot transfer. Improvements with target labels measure supervised recalibration and should not be compared to label-free adaptation methods as equivalent information budgets.

## 7. Later-day control

An additional chronological split of the paired seven-class dataset tests the sensitivity of the cross-device result to the evaluation protocol.

In [ ]:
chrono_train = np.flatnonzero(paired.time.to_numpy() < np.datetime64("2026-10-21"))
chrono_test = np.flatnonzero(paired.time.to_numpy() >= np.datetime64("2026-10-21"))
assert not np.intersect1d(chrono_train, chrono_test).size
assert all(set(y[ix]) == set(range(7)) for ix in (chrono_train, chrono_test))
chronorows = []
for src, tgt in ((171, 172), (172, 171)):
    ss, tt = tree_stats(X[src]), tree_stats(X[tgt])
    model = ExtraTreesClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
    model.fit(ss[chrono_train], y[chrono_train])
    chronorows.append({"direction": f"{src}->{tgt}", "method": "source_only_trees", "budget": 0,
                       "seed": 42, "n_train": len(chrono_train), "n_test": len(chrono_test),
                       **summary_metrics(y[chrono_test], model.predict(tt[chrono_test]))})
    for seed in CALIB_SEEDS:
        rng = np.random.default_rng(seed)
        order = {k: rng.permutation(chrono_train[y[chrono_train] == k]) for k in range(7)}
        for budget in BUDGETS:
            fit = np.concatenate([order[k][:budget] for k in range(7)])
            assert len(fit) == budget * 7
            for method in ("target_only_trees", "pooled_trees"):
                model = ExtraTreesClassifier(n_estimators=300, class_weight="balanced",
                                             random_state=seed, n_jobs=-1)
                if method == "target_only_trees":
                    model.fit(tt[fit], y[fit])
                else:
                    weights = np.r_[np.full(len(chrono_train), .5/len(chrono_train)),
                                    np.full(len(fit), .5/len(fit))]
                    model.fit(np.vstack([ss[chrono_train], tt[fit]]),
                              np.r_[y[chrono_train], y[fit]], sample_weight=weights)
                chronorows.append({"direction": f"{src}->{tgt}", "method": method,
                                   "budget": budget, "seed": seed,
                                   "n_train": len(chrono_train), "n_test": len(chrono_test),
                                   **summary_metrics(y[chrono_test], model.predict(tt[chrono_test]))})
chrono = pd.DataFrame(chronorows)
chrono.to_csv(OUT / "chronological_control.csv", index=False)
print("Chronological test", paired.time.iloc[chrono_test].min(), "...", paired.time.iloc[chrono_test].max())
print(chrono.groupby(["direction", "method", "budget"]).bacc.mean().round(3).to_string())

## 8. Paired mixture task

Treat cards 129/130 as a separate eight-composition multi-label problem. Report it independently of the seven-class primary task.

In [ ]:
MIX_OUT = OUT / "mixtures_129_130"
MIX_GASES = ("acetone", "isopropanol", "pentane")
MIX_EXISTS = RUN_MIXTURES and all((DATA_DIR / fname).exists() for fname in MIX_ARCHIVES.values())


def mixture_label(raw):
    raw = str(raw).lower().replace(" ", "").split("_d", 1)[0]
    if raw in {"air", "none"}:
        return (0, 0, 0)
    components = raw.split("+")
    names = [re.match(r"^([a-z]+)", c).group(1) for c in components if re.match(r"^([a-z]+)", c)]
    if len(names) != len(components) or len(names) != len(set(names)) or not set(names) <= set(MIX_GASES):
        raise ValueError(f"Unrecognized mixture label: {raw}")
    return tuple(int(gas in names) for gas in MIX_GASES)


def load_mixture_card(card):
    source = EsmellDataset(root_path=str(DATA_DIR), subfolders=MIX_ARCHIVES[card],
                           csv_repair="strict", archive_cache="disk", preload=True)
    pipeline = source >> Chain(
        DefaultCSVPreprocessing(), SimplifyLabels(), SelectRChannels(include=CHANNELS),
        UserLabelCycleSegments(),
        CoolingHeatingSegments(temp_col="T_mean", smooth_window=3, eps=.2),
        CanonicalTempAlign(n_grid=200, temp_range=(175.,390.), normalize=False, conductance=True),
        CanonicalSubtractionFeatures(features=(HEAT, (HEAT, COOL))),
    )
    rows, samples, errors = [], [], []
    for sid in tqdm(source.ids, desc=f"Card {card}"):
        info = source.metadata(sid)
        try:
            vec = mixture_label(info["classname"])
            x = np.asarray(pipeline.features(sid), dtype=np.float32)
            if x.shape != (4, 200) or not np.isfinite(x).all():
                raise ValueError(f"Invalid feature tensor: {x.shape}")
            rows.append({"id": str(info["filename"]), "time": pd.Timestamp(info["datetime"]),
                         "raw_label": str(info["classname"]), "composition": ''.join(map(str,vec)),
                         "x_idx": len(samples)})
            samples.append(x)
        except Exception as e:
            errors.append({"card": card, "file": info["filename"], "reason": str(e)[:200]})
    if not samples:
        raise RuntimeError(f"No valid samples for card {card}")
    return pd.DataFrame(rows), np.stack(samples), errors


if not MIX_EXISTS:
    print("E2-B skipped (RUN_MIXTURES=False or missing files):", [p for p in MIX_ARCHIVES.values() if not (DATA_DIR / p).exists()])
else:
    MIX_OUT.mkdir(parents=True, exist_ok=True)
    mix_cards = {}
    mix_qc = []
    for c in MIX_ARCHIVES:
        info, arr, errors = load_mixture_card(c)
        mix_cards[c] = (info, arr)
        mix_qc.extend(errors)
    pd.DataFrame(mix_qc, columns=["card", "file", "reason"]).to_csv(MIX_OUT / "qc_errors.csv",index=False)
    left = mix_cards[129][0].rename(columns={"id":"id_129", "x_idx":"ix_129", "raw_label":"label_129"})
    right = mix_cards[130][0].rename(columns={"id":"id_130", "x_idx":"ix_130", "raw_label":"label_130"})
    mix_pairs = pd.merge_asof(left.sort_values('time'), right.sort_values('time'),
                               on='time', by='composition', direction='nearest',
                               tolerance=pd.Timedelta(seconds=2)).dropna(subset=["ix_130"]).copy()
    # Validity: reject pairs with nominally different concentrations/recipe.
    mix_pairs = mix_pairs.loc[mix_pairs.label_129.eq(mix_pairs.label_130)].copy()
    assert mix_pairs.id_129.is_unique and mix_pairs.id_130.is_unique, 'Non-unique mixture pairs'
    mix_pairs = mix_pairs.sort_values('time').reset_index(drop=True)
    mix_pairs['pair_id'] = ['mix_%04d' % i for i in range(len(mix_pairs))]
    assert len(mix_pairs) > 1500, f'Unexpectedly few pairs: {len(mix_pairs)}; inspect QC and labels'
    assert mix_pairs.time.max() <= pd.Timestamp('2026-09-29'), 'Unpaired 130-only tail in E2-B'
    mix_y = np.array([[int(v) for v in comp] for comp in mix_pairs.composition], dtype=int)
    mix_X = {c: mix_cards[c][1][mix_pairs[f'ix_{c}'].astype(int).to_numpy()] for c in (129,130)}
    mix_pairs.to_csv(MIX_OUT / 'pairs.csv', index=False)
    print('E2-B matched pairs:', len(mix_pairs), '\n',mix_pairs.composition.value_counts().to_string())

In [ ]:
if MIX_EXISTS:
    groups = mix_pairs.composition.to_numpy()
    outer = StratifiedShuffleSplit(n_splits=1, test_size=.20, random_state=SPLIT_SEED)
    mix_dev_ix, mix_test_ix = next(outer.split(np.arange(len(groups)), groups))
    assert not np.intersect1d(mix_dev_ix, mix_test_ix).size
    mix_split_rows = [{"role": role, "pair_id": mix_pairs.pair_id.iloc[i], "seed": SPLIT_SEED, "budget": 0}
                      for role, inds in (("dev",mix_dev_ix),("test",mix_test_ix)) for i in inds]
    comp_set = tuple(sorted(set(groups)))
    assert all(np.sum(groups[mix_dev_ix] == c) >= max(BUDGETS) for c in comp_set), \
        'Insufficient samples per composition for the declared budgets'
    assert all(np.sum(groups[mix_test_ix] == c) > 0 for c in comp_set)
    assert all(len(np.unique(mix_y[mix_test_ix, k])) == 2 for k in range(3)), \
        'Each gas must have both positive and negative test examples' 

    # Per-label mean of sensitivity and specificity = macro BAcc over three gas labels.
    def mix_metrics(true, pred):
        assert true.shape == pred.shape and true.shape[1] == 3
        bac = [balanced_accuracy_score(true[:,i],pred[:,i]) for i in range(3)]
        return {"macro_bacc": float(np.mean(bac)),
                "macro_f1": float(f1_score(true,pred,average="macro",zero_division=0)),
                "exact_match": float(accuracy_score(true,pred))}

    def mix_model(seed):
        return MultiOutputClassifier(ExtraTreesClassifier(n_estimators=300, min_samples_leaf=1,
                              class_weight="balanced",random_state=seed,n_jobs=-1))

    mix_results, mix_predictions = [], []
    def mix_record(src, tgt, method, budget, seed, pred, n_target):
        stats = mix_metrics(mix_y[mix_test_ix], pred)
        pure = mix_y[mix_test_ix].sum(axis=1) <= 1
        stats['exact_match_pure'] = float(accuracy_score(mix_y[mix_test_ix][pure],pred[pure])) if pure.any() else np.nan
        stats['exact_match_mixtures'] = float(accuracy_score(mix_y[mix_test_ix][~pure],pred[~pure])) if (~pure).any() else np.nan
        mix_results.append(dict(direction=f'{src}->{tgt}', method=method, budget_per_composition=budget,
                                seed=seed, n_target_labels=n_target,n_test=len(mix_test_ix),**stats))
        for i,v in zip(mix_test_ix, pred):
            mix_predictions.append(dict(direction=f'{src}->{tgt}',method=method,budget_per_composition=budget,
                                        seed=seed,pair_id=mix_pairs.pair_id.iloc[i],
                                        true=''.join(map(str,mix_y[i])),pred=''.join(map(str,v))))


    class MixCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = EsmellEncoder(4)
            self.classifier = nn.Sequential(nn.Linear(128, 128), nn.ReLU(),
                                            nn.Dropout(.2), nn.Linear(128, 3))
        def forward(self, x):
            return self.classifier(self.encoder(x))

    def mix_loader(x, inds, seed, shuffle=False):
        return DataLoader(TensorDataset(torch.as_tensor(x[inds], dtype=torch.float32),
                                        torch.as_tensor(mix_y[inds], dtype=torch.float32)),
                          batch_size=32 if shuffle else 128, shuffle=shuffle,
                          generator=torch.Generator().manual_seed(seed))

    def mix_predict(model, x, inds):
        model.eval()
        with torch.no_grad():
            return np.concatenate([(torch.sigmoid(model(xb.to(DEVICE))) >= .5).to(torch.int64).cpu().numpy()
                                   for xb, _ in mix_loader(x, inds, seed=42)])

    # Source-only checkpoint selected only on source validation.
    mix_train_loc, mix_val_loc = next(StratifiedShuffleSplit(n_splits=1, test_size=.25,
                                    random_state=SPLIT_SEED).split(mix_dev_ix, groups[mix_dev_ix]))
    mix_source_train, mix_source_val = mix_dev_ix[mix_train_loc], mix_dev_ix[mix_val_loc]

    def mix_source_cnn(x):
        set_seed(SPLIT_SEED)
        model = MixCNN().to(DEVICE)
        positives = mix_y[mix_source_train].sum(axis=0)
        assert positives.min() > 0
        weights = torch.tensor((len(mix_source_train) - positives) / positives,
                               device=DEVICE, dtype=torch.float32)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        best, stale, state = -1., 0, None
        for _ in range(EPOCHS):
            model.train()
            for xb,yb in mix_loader(x, mix_source_train,seed=SPLIT_SEED,shuffle=True):
                optimizer.zero_grad()
                nn.functional.binary_cross_entropy_with_logits(model(xb.to(DEVICE)),
                                                  yb.to(DEVICE),pos_weight=weights).backward()
                optimizer.step()
            score = f1_score(mix_y[mix_source_val], mix_predict(model,x,mix_source_val),
                             average='macro',zero_division=0)
            if score > best:
                best, state, stale = score, copy.deepcopy(model.state_dict()), 0
            else:
                stale += 1
                if stale >= PATIENCE:
                    break
        model.load_state_dict(state)
        return model

    def mix_target_cnn(x, fit, seed, pretrained=None):
        set_seed(seed)
        model = copy.deepcopy(pretrained).to(DEVICE) if pretrained is not None else MixCNN().to(DEVICE)
        rate = 1e-4 if pretrained is not None else 1e-3
        epochs = ADAPT_EPOCHS if pretrained is not None else TARGET_ONLY_EPOCHS
        optimizer = torch.optim.Adam(model.parameters(), lr=rate, weight_decay=1e-4)
        for _ in range(epochs):
            model.train()
            if pretrained is not None:
                for layer in model.modules():
                    if isinstance(layer, nn.BatchNorm1d): layer.eval()
            for xb,yb in mix_loader(x,fit,seed=seed,shuffle=True):
                optimizer.zero_grad()
                nn.functional.binary_cross_entropy_with_logits(model(xb.to(DEVICE)),yb.to(DEVICE)).backward()
                optimizer.step()
        return model

    for src,tgt in ((129,130),(130,129)):
        source_raw, target_raw = mix_X[src], mix_X[tgt]
        src_net = mix_source_cnn(source_raw)
        mix_record(src,tgt,'source_only_cnn',0,SPLIT_SEED,
                   mix_predict(src_net, target_raw, mix_test_ix),0)
        source_x, target_x = tree_stats(source_raw),tree_stats(target_raw)
        base = mix_model(SPLIT_SEED)
        base.fit(source_x[mix_dev_ix],mix_y[mix_dev_ix])
        mix_record(src,tgt,'source_only_trees',0,SPLIT_SEED,base.predict(target_x[mix_test_ix]),0)
        full = mix_model(SPLIT_SEED)
        full.fit(target_x[mix_dev_ix],mix_y[mix_dev_ix])
        mix_record(src,tgt,'target_reference',-1,SPLIT_SEED,full.predict(target_x[mix_test_ix]),len(mix_dev_ix))
        for seed in CALIB_SEEDS:
            rng = np.random.default_rng(seed)
            order = {c: rng.permutation(mix_dev_ix[groups[mix_dev_ix] == c]) for c in comp_set}
            for budget in BUDGETS:
                fit = np.concatenate([order[c][:budget] for c in comp_set])
                assert len(fit) == len(comp_set)*budget and not np.intersect1d(fit, mix_test_ix).size
                mix_split_rows.extend(dict(role='target_calibration',pair_id=mix_pairs.pair_id.iloc[i],
                                            seed=seed,budget=budget) for i in fit)
                fine = mix_target_cnn(target_raw,fit,seed,pretrained=src_net)
                mix_record(src,tgt,'finetune_full_cnn',budget,seed,
                           mix_predict(fine,target_raw,mix_test_ix),len(fit))
                del fine
                fresh = mix_target_cnn(target_raw,fit,seed)
                mix_record(src,tgt,'target_only_cnn',budget,seed,
                           mix_predict(fresh,target_raw,mix_test_ix),len(fit))
                del fresh
                target = mix_model(seed)
                target.fit(target_x[fit],mix_y[fit])
                mix_record(src,tgt,'target_only_trees',budget,seed,target.predict(target_x[mix_test_ix]),len(fit))
                # sklearn MultiOutputClassifier.fit forwards sample_weight to each estimator.
                pooled = mix_model(seed)
                weights = np.r_[np.full(len(mix_dev_ix),.5/len(mix_dev_ix)),
                                np.full(len(fit),.5/len(fit))]
                pooled.fit(np.vstack([source_x[mix_dev_ix], target_x[fit]]),
                           np.vstack([mix_y[mix_dev_ix],mix_y[fit]]),sample_weight=weights)
                mix_record(src,tgt,'pooled_trees',budget,seed,pooled.predict(target_x[mix_test_ix]),len(fit))
    pd.DataFrame(mix_results).to_csv(MIX_OUT/'scores.csv',index=False)
    pd.DataFrame(mix_predictions).to_csv(MIX_OUT/'predictions.csv',index=False)
    pd.DataFrame(mix_split_rows).to_csv(MIX_OUT/'splits.csv',index=False)
    mix_summary = pd.DataFrame(mix_results).groupby(['direction','method','budget_per_composition'],as_index=False).agg(
        macro_bacc_mean=('macro_bacc','mean'), macro_bacc_sd=('macro_bacc','std'),
        macro_f1_mean=('macro_f1','mean'),exact_match_mean=('exact_match','mean'),
        exact_match_pure=('exact_match_pure','mean'),exact_match_mixtures=('exact_match_mixtures','mean'),
        n_target_labels=('n_target_labels','first'))
    mix_summary.to_csv(MIX_OUT/'summary.csv',index=False)
    print('\nE2-B results:')
    print(mix_summary.round(3).to_string(index=False))

## Outputs

The score tables, matched-pair manifest, individual predictions, and calibration curves are retained in `results/E2/`.